# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    # TODO: loop up to `tries` times
    # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
    # TODO:   raise on a status that is not in RETRY_STATUSES
    # TODO:   otherwise sleep (increasing each time) and try again
    # TODO: raise if every attempt failed
    last_problem = None
    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.ConnectionError, requests.Timeout) as e:
            last_problem = repr(e)              # network blip: worth retrying
        else:
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])
            if r.status_code not in RETRY_STATUSES:
                # 400/404 etc. are our bug -- fail immediately, no retry
                raise requests.HTTPError(
                    f'{r.status_code} from Open-Meteo: {r.text[:200]}', response=r)
            last_problem = f'HTTP {r.status_code}'
        if attempt < tries - 1:
            time.sleep(2 ** attempt)            # 1s, 2s, 4s... increasing delay
    raise RuntimeError(f'Open-Meteo failed after {tries} attempts ({last_problem})')

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
DATES = ['2026-09-05', '2026-10-03']
used_fallback = False

wx = None
pulled_at = pd.Timestamp.now(tz='America/New_York')

try:
    raw = get_weather(DATES[0], DATES[1])        # one request covering both dates
    wx = raw[raw['time'].isin(DATES)].reset_index(drop=True)
except (requests.ConnectionError, requests.Timeout, RuntimeError) as e:
    print(f'Live pull failed: {e} -> using fallback sample')
    used_fallback = True

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'
if used_fallback:
    wx = FALLBACK.copy()

wx['source'] = 'fallback sample' if used_fallback else 'open-meteo'

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
# TODO
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})
wx['date'] = pd.to_datetime(wx['date'])
wx = wx[['date', 'temp_max', 'precip_mm', 'source']]
wx

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
# assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
# assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
# assert pd.api.types.is_datetime64_any_dtype(wx['date'])
# assert wx['precip_mm'].notna().all(), 'missing precipitation'
# assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
# TODO
wet = wx.loc[wx['precip_mm'].idxmax()]
dry = wx.loc[wx['precip_mm'].idxmin()]
diff = wet['precip_mm'] - dry['precip_mm']

if diff == 0:
    print(f"Both home weekends got the same rainfall ({wet['precip_mm']:.1f} mm).")
else:
    print(f"{wet['date']:%A, %B %d} was the wetter home weekend: "
          f"{wet['precip_mm']:.1f} mm of rain vs {dry['precip_mm']:.1f} mm on "
          f"{dry['date']:%B %d}, which is {diff:.1f} mm ({diff / 25.4:.2f} in) more.")

if used_fallback:
    print('NOTE: these figures are sample data, not real observations.')

Saturday, October 03 was the wetter home weekend: 14.7 mm of rain vs 2.5 mm on September 05, which is 12.2 mm (0.48 in) more.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
# TODO
print('Data source :', BASE)
print('Coordinates :', '38.03, -78.51 (Charlottesville, VA)')
print('Date range  :', f'{DATES[0]} to {DATES[1]} (kept {", ".join(DATES)})')
print('Pulled at   :', pulled_at.strftime('%Y-%m-%d %H:%M %Z'))
print('Data status :', 'FALLBACK SAMPLE, not real' if used_fallback else 'real (Open-Meteo archive)')

Data source : https://archive-api.open-meteo.com/v1/archive
Coordinates : 38.03, -78.51 (Charlottesville, VA)
Date range  : 2026-09-05 to 2026-10-03 (kept 2026-09-05, 2026-10-03)
Pulled at   : 2026-10-11 22:15 EDT
Data status : real (Open-Meteo archive)


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather returned an empty DataFrame on failure, nothing would error: merging it onto 200 days of sales just fills the rain column with NaN, and any fillna(0) along the way turns that into “0 mm of rain every day.” The chart would then be blank or a flat line, and the apparent finding would be that rain has no effect on revenue. Nobody would catch it because every cell runs green and “no effect” is a plausible, boring result, so the failure in the fetch surfaces three steps later as a conclusion with nothing pointing back to its cause. Raising an error instead makes the problem visible at the source, which is also why the Part 5 asserts check for the right row count and missing values.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
# stretch
cities = {'Charlottesville': (38.03, -78.51), 'Orlando': (28.54, -81.38)}
frames = []
for city, (lat, lon) in cities.items():
    raw = get_weather(DATES[0], DATES[1], lat=lat, lon=lon)
    f = raw[raw['time'].isin(DATES)].copy()
    f['city'] = city
    f['source'] = 'open-meteo'
    frames.append(f)
    time.sleep(1)                                # be polite to the free API

both = (pd.concat(frames, ignore_index=True)
          .rename(columns={'time': 'date', 'temperature_2m_max': 'temp_max',
                           'precipitation_sum': 'precip_mm'}))
both['date'] = pd.to_datetime(both['date'])
both = both[['city', 'date', 'temp_max', 'precip_mm', 'source']]
assert len(both) == 4
both

,city,date,temp_max,precip_mm,source
0,Charlottesville,2026-09-05,29.4,2.5,open-meteo
1,Charlottesville,2026-10-03,22.0,14.7,open-meteo
2,Orlando,2026-09-05,32.1,3.9,open-meteo
3,Orlando,2026-10-03,30.4,14.5,open-meteo
